In [1]:
import pandas as pd
locations = pd.read_excel(
    'ArchivosExcel/SL0202_Replacement_Locations_test.xlsx',
    skiprows=1
)
locations.head()

,Location_ID,Location_Name,Latitude,Longitude
0,1,Empire State Building,"40°44'54.4""N","73°59'08.4""W"
1,2,Statue of Liberty,"40°41'21.2""N","74°02'40.2""W"
2,3,White House,"38°53'52.6""N","77°02'11.6""W"
3,4,Golden Gate Bridge,"37°49'11.0""N","122°28'43.0""W"
4,5,Space Needle,"47°37'13.0""N","122°20'57.0""W"


In [2]:
#creating a sub-dataframe with the coordinates only
latlongs = locations[['Latitude', 'Longitude']].copy()
latlongs.head()

,Latitude,Longitude
0,"40°44'54.4""N","73°59'08.4""W"
1,"40°41'21.2""N","74°02'40.2""W"
2,"38°53'52.6""N","77°02'11.6""W"
3,"37°49'11.0""N","122°28'43.0""W"
4,"47°37'13.0""N","122°20'57.0""W"


In [3]:
import re
# Define a helper function to convert Degree-Minute-Second (DMS) to decimal
def dms_to_dd(dms_str):
    # Convert the input to string to handle potential non-string values
    dms_str = str(dms_str)
    parts = re.split('[°\'"]+', dms_str)

    # Handle cases where the input is not in DMS format or is empty
    if len(parts) != 4:
        return float('nan')  # Return NaN for invalid inputs

    degrees = float(parts[0])
    minutes = float(parts[1])
    seconds = float(parts[2])
    direction = parts[3].strip()

    dd = degrees + minutes / 60 + seconds / 3600
    if direction in ['S', 'W']:
        dd *= -1  # South and West are negative
    return dd

# Apply the conversion to both Latitude and Longitude columns using .loc to avoid warnings
latlongs.loc[:, 'Latitude_dec'] = latlongs['Latitude'].apply(dms_to_dd)
latlongs.loc[:, 'Longitude_dec'] = latlongs['Longitude'].apply(dms_to_dd)

In [4]:
# Display the sample result
latlongs[['Latitude', 'Longitude', 'Latitude_dec', 'Longitude_dec']].head()

,Latitude,Longitude,Latitude_dec,Longitude_dec
0,"40°44'54.4""N","73°59'08.4""W",40.748444,-73.985667
1,"40°41'21.2""N","74°02'40.2""W",40.689222,-74.044500
2,"38°53'52.6""N","77°02'11.6""W",38.897944,-77.036556
3,"37°49'11.0""N","122°28'43.0""W",37.819722,-122.478611
4,"47°37'13.0""N","122°20'57.0""W",47.620278,-122.349167


In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from urllib.parse import quote_plus
import pandas as pd
import html
import time

# Start a fresh Chrome session. No Google account or API key is required.
options = webdriver.ChromeOptions()

options.add_argument('--headless=new')

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 12)

In [6]:
# Helper: extract the address currently displayed by Google Maps
def get_address_from_maps(driver):
    selectors = [
        '[data-item-id="address"]',
        'button[aria-label^="Address:"]',
        'button[aria-label^="Dirección:"]'
    ]

    for selector in selectors:
        try:
            element = WebDriverWait(driver, 4).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, selector))
            )
            value = element.get_attribute('aria-label') or element.text
            if value:
                value = re.sub(r'^(Address|Dirección):\s*', '', value).strip()
                if value:
                    return value
        except Exception:
            pass

    return 'Address not found'


# Helper: try to find a U.S.-style county name in the Maps page.
# Unlike the API, the Google Maps UI does not expose structured address components,
# so this is best-effort and may return 'County not found'.
def extract_county_from_maps(driver, address=''):
    try:
        body_text = driver.find_element(By.TAG_NAME, 'body').text
    except Exception:
        body_text = ''

    searchable = f'{address}\n{body_text}'
    county_match = re.search(
        r'\b([A-Z][A-Za-z .\'-]{1,50} County)\b',
        searchable
    )
    if county_match:
        return county_match.group(1).strip()

    # Fallback: Maps sometimes keeps additional location text in its page source.
    try:
        source = html.unescape(driver.page_source)
        county_match = re.search(
            r'\b([A-Z][A-Za-z .\'-]{1,50} County)\b',
            source
        )
        if county_match:
            return county_match.group(1).strip()
    except Exception:
        pass

    return 'County not found'


# Function to reverse-locate coordinates using the Google Maps website with Selenium
def get_county_google(lat, lon):
    try:
        # Check if latitude and longitude are valid
        if pd.isna(lat) or pd.isna(lon):
            return {'County': 'Invalid Coordinates', 'Google_Address': 'Invalid Coordinates'}
        if not (-90 <= lat <= 90) or not (-180 <= lon <= 180):
            return {'County': 'Invalid Lat/Long Range', 'Google_Address': 'Invalid Lat/Long Range'}

        print(f'Processing Lat: {lat}, Lon: {lon}')

        # Search the coordinates directly in Google Maps.
        # This Maps URL does not require a Google Cloud API key.
        query = quote_plus(f'{lat},{lon}')
        driver.get(f'https://www.google.com/maps/search/?api=1&query={query}')

        # Wait until the Maps interface is present before reading it.
        WebDriverWait(driver, 12).until(
            EC.presence_of_element_located((By.TAG_NAME, 'body'))
        )
        time.sleep(1.5)

        address = get_address_from_maps(driver)
        county = extract_county_from_maps(driver, address)

        print(f'  Address: {address}')
        print(f'  County: {county}')

        return {'County': county, 'Google_Address': address}

    except Exception as e:
        print(f'Error: {e}')
        return {'County': 'Error', 'Google_Address': 'Error'}


In [7]:
# Apply the Selenium-based Google Maps lookup to each lat-long pair
results = latlongs.apply(
    lambda row: get_county_google(row['Latitude_dec'], row['Longitude_dec']),
    axis=1
)

# Keep the original County logic and also store the address shown by Maps
latlongs.loc[:, 'County'] = results.apply(lambda x: x['County'])
latlongs.loc[:, 'Google_Address'] = results.apply(lambda x: x['Google_Address'])

Processing Lat: 40.748444444444445, Lon: -73.98566666666667
  Address: Address not found
  County: County not found
Processing Lat: 40.68922222222222, Lon: -74.0445
  Address: Address not found
  County: County not found
Processing Lat: 38.89794444444444, Lon: -77.03655555555555
  Address: Address not found
  County: County not found
Processing Lat: 37.819722222222225, Lon: -122.4786111111111
  Address: Address not found
  County: County not found
Processing Lat: 47.62027777777778, Lon: -122.34916666666666
  Address: Address not found
  County: County not found
Processing Lat: 41.87883333333333, Lon: -87.63597222222224
  Address: Address not found
  County: County not found
Processing Lat: 30.27472222222222, Lon: -97.74027777777778
  Address: Address not found
  County: County not found
Processing Lat: 25.793333333333337, Lon: -80.29055555555556
  Address: Address not found
  County: County not found


In [8]:
# Display the result
latlongs[['Latitude', 'Longitude', 'Latitude_dec', 'Longitude_dec', 'Google_Address', 'County']].head()

,Latitude,Longitude,Latitude_dec,Longitude_dec,Google_Address,County
0,"40°44'54.4""N","73°59'08.4""W",40.748444,-73.985667,Address not found,County not found
1,"40°41'21.2""N","74°02'40.2""W",40.689222,-74.044500,Address not found,County not found
2,"38°53'52.6""N","77°02'11.6""W",38.897944,-77.036556,Address not found,County not found
3,"37°49'11.0""N","122°28'43.0""W",37.819722,-122.478611,Address not found,County not found
4,"47°37'13.0""N","122°20'57.0""W",47.620278,-122.349167,Address not found,County not found


In [9]:
#adding the newly generated columns into the original dataset
locations['Google_Address'] = latlongs['Google_Address']
locations['County'] = latlongs['County']
locations.head()

,Location_ID,Location_Name,Latitude,Longitude,Google_Address,County
0,1,Empire State Building,"40°44'54.4""N","73°59'08.4""W",Address not found,County not found
1,2,Statue of Liberty,"40°41'21.2""N","74°02'40.2""W",Address not found,County not found
2,3,White House,"38°53'52.6""N","77°02'11.6""W",Address not found,County not found
3,4,Golden Gate Bridge,"37°49'11.0""N","122°28'43.0""W",Address not found,County not found
4,5,Space Needle,"47°37'13.0""N","122°20'57.0""W",Address not found,County not found


In [10]:
# Close the Selenium browser once all locations have been processed
driver.quit()

#saving the updated dataframe to a file
locations.to_excel(
    'ArchivosExcel/SL0202_Replacement_Locations_with_County_Selenium.xlsx',
    index=False
)